### Setup

In [12]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
from pyproj import Transformer
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
from shapely.ops import transform

### Read City Boundaries

In [ ]:
B8_path = r"PATH"
B11_path = r"PATH"

#BERLIN_gdf= gpd.read_file(r"PATH")
PARIS_gdf = gpd.read_file(r"PATH")
#LONDON_gdf = gpd.read_file(r"PATH")

PARIS_gdf.set_crs("EPSG:4326", inplace=True)

In [ ]:
B8 = rxr.open_rasterio(B8_path).squeeze()
B11 = rxr.open_rasterio(B11_path).squeeze()

In [ ]:
cities = {
    "Paris": {
        "swir": B11,
        "nir":  B8,
        "gdf":  PARIS_gdf,
        "utm_epsg": 32631
    }
}

In [ ]:
def compute_ndbi(swir, nir):
    swir = swir.astype("float32")
    nir  = nir.astype("float32")
    return (swir - nir) / (swir + nir + 1e-6)

def clip_raster_to_city(raster_da, gdf):
    gdf_proj = gdf.to_crs(raster_da.rio.crs)
    clipped = raster_da.rio.clip(gdf_proj.geometry, gdf_proj.crs, drop=True)
    return clipped

In [ ]:

def plot_ndbi(ndbi_da, title, figsize=(8, 8)):
    fig, ax = plt.subplots(figsize=figsize)

    ndbi_da.plot(ax=ax, robust=True)

    ax.set_aspect("equal", adjustable="box")
    ax.autoscale(False) 

    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
B11_clip = clip_raster_to_city(B11, PARIS_gdf)
B8_clip = clip_raster_to_city(B8, PARIS_gdf)

ndbi = compute_ndbi(B11_clip, B8_clip)
ndbi.name = "NDBI"
plot_ndbi(ndbi, "Paris NDBI")


In [ ]:

def builtup_percent_from_ndbi(ndbi_da, threshold=0.0):
    valid = np.isfinite(ndbi_da.values)
    built = valid & (ndbi_da.values > threshold)
    pct = 100.0 * built.sum() / valid.sum()
    return pct, built.sum(), valid.sum()

pct, built_n, total_n = builtup_percent_from_ndbi(ndbi, threshold=0.0)
print(f"Built-up area (NDBI > 0.0): {pct:.2f}%  ({built_n}/{total_n} pixels)")


### Clip Band to Raster

In [ ]:
def clip_band_safe(band_path, gdf, utm_epsg):
    with rasterio.open(band_path) as src:
        raster_crs = f"EPSG:{utm_epsg}"
        transformer = Transformer.from_crs("EPSG:4326", raster_crs, always_xy=True)
        gdf_proj = gdf.copy()
        gdf_proj["geometry"] = gdf_proj["geometry"].apply(lambda geom: transform(transformer.transform, geom))
        
        clipped, out_transform = mask(src, gdf_proj.geometry.values, crop=True)
        band = clipped[0].astype("float32")
        
        meta = src.meta.copy()
        meta.update({
            "height": band.shape[0],
            "width": band.shape[1],
            "transform": out_transform,
            "count": 1,
            "dtype": "float32",
            "crs": raster_crs 
        })
    return band, meta



### Compute NDBI 

In [ ]:
def compute_ndbi(swir, nir):
    swir = swir.astype("float32")
    nir = nir.astype("float32")
    ndbi = (swir - nir) / (swir + nir + 1e-6)
    return ndbi


In [ ]:
for city, data in cities.items():
    print(f"Processing {city}...")

    swir_clip = clip_raster_to_city(
        data["swir"],
        data["gdf"],
        data["utm_epsg"]
    )

    nir_clip = clip_raster_to_city(
        data["nir"],
        data["gdf"],
        data["utm_epsg"]
    )

    ndbi = compute_ndbi(swir_clip, nir_clip)

### Plot NDVBI

In [ ]:
import matplotlib.pyplot as plt

def plot_ndbi(ndbi, title, factor=10):
    ndbi_small = ndbi[::factor, ::factor]
    plt.figure(figsize=(6, 6))
    plt.imshow(ndbi_small, cmap="RdYlGn", vmin=-1, vmax=1)
    plt.colorbar(label="NDBI")
    plt.title(title)
    plt.axis("off")
    plt.show()

plot_ndbi(ndbi, f"Paris NDBI")
